# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Saadali880/flyrank-ml-internship-saad/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

# ## 1. My rule and its reason codes

### Plain Words Rule Description
Prioritize pages that are **stale** (not updated in 90+ days), **visible** (at least 500 GSC impressions in the trailing 90 days), and in **striking distance** (GSC average position between 3 and 15). These represent high-value search assets that are at risk of content decay and should be refreshed to protect and recover their rankings.

### Rule Specifications
- **Reason Code**: `stale_striking_visible` (Exactly ONE reason code for matched pages)
- **Action Label**: `refresh` (Exactly ONE action label for matched pages)
- **Score**: Continuous baseline score combining staleness rank ($0.4$), visibility rank ($0.4$), and position opportunity ($0.2$) for matched pages, and $0.0$ for non-matched pages.

### Signal Audit & Verdicts
Before building our rule, we audit two key signals against `is_declining_label` (decline defined as GSC impressions dropping by >20% in the last 30 days vs. previous 30 days):
1. **Signal 1: Staleness (`days_since_last_update`)** (linked to FlyRank's refresh flags) — **Verdict: MIXED**
   - *Explanation*: The decline rate rises from 51.1% (fresh content, 0-30 days) to 58.9% (31-90 days) and peaks at 61.1% (91-180 days). However, it drops back down to 47.1% for very stale content (181+ days). This non-linear trend suggests that while age increases decline risk initially, the oldest surviving pages are highly stable 'evergreen' content (survivorship bias).
2. **Signal 2: Word Count (`word_count`)** (proxy for thin content) — **Verdict: OPPOSITE**
   - *Explanation*: Traditional rules assume thin pages are low quality and prone to decline. However, the data reveals that thin content (<=1000 words) has a decline rate of only 20.7%, whereas longer articles (1000-2000, 2000-3500, 3500+) have a much higher decline rate of 55-60%. Thus, targeting thin pages for decline risk is contradicted by the data.


In [1]:
import pandas as pd
import numpy as np

# 1. Load the processed feature vector
df = pd.read_csv("D:/Flyrank/data/processed/refresh_feature_vector.csv")
print(f"Loaded feature vector with {len(df)} rows and {len(df.columns)} columns.")
print(f"Base rate of decline: {df['is_declining_label'].mean():.3f}\n")

# 2. Check Signal 1: Staleness (days_since_last_update)
print("=== Signal 1 Check: days_since_last_update (freshness_tier) ===")
signal1_table = df.groupby('freshness_tier').agg(
    n=('is_declining_label', 'count'),
    decline_rate=('is_declining_label', 'mean')
).reset_index()
print(signal1_table.to_string(index=False))
print("\nVerdict: MIXED")
print("Explanation: Decline rate rises from 51.1% to 61.1% (91-180 days) but drops to 47.1% at 181+ days.\n")

# 3. Check Signal 2: Word Count (word_count)
print("=== Signal 2 Check: word_count_bin ===")
df['word_count_bin'] = pd.cut(
    df['word_count'], 
    bins=[-1, 0, 1000, 2000, 3500, 100000], 
    labels=['missing', '<=1000', '1000-2000', '2000-3500', '3500+']
)
signal2_table = df.groupby('word_count_bin', observed=False).agg(
    n=('is_declining_label', 'count'),
    decline_rate=('is_declining_label', 'mean')
).reset_index()
print(signal2_table.to_string(index=False))
print("\nVerdict: OPPOSITE")
print("Explanation: Thin pages (<=1000 words) have a 20.7% decline rate, much lower than longer pages (55-60%).")


Loaded feature vector with 30000 rows and 52 columns.
Base rate of decline: 0.542

=== Signal 1 Check: days_since_last_update (freshness_tier) ===
freshness_tier     n  decline_rate
          0-30 20480      0.511377
          181+   174      0.471264
         31-90   175      0.588571
        91-180  9171      0.611057

Verdict: MIXED
Explanation: Decline rate rises from 51.1% to 61.1% (91-180 days) but drops to 47.1% at 181+ days.

=== Signal 2 Check: word_count_bin ===
word_count_bin     n  decline_rate
       missing  7699      0.465385
        <=1000   973      0.206578
     1000-2000  3781      0.555409
     2000-3500 11262      0.588439
         3500+  6285      0.596818

Verdict: OPPOSITE
Explanation: Thin pages (<=1000 words) have a 20.7% decline rate, much lower than longer pages (55-60%).


## 2. Build the ranked queue (writes the CSV)

We calculate the baseline score using only leakage-free, historical variables. The score is only non-zero for matched rows. We then rank the queue and export it to `work/outputs/baseline_action_score.csv`.


In [2]:
import os

# Helper functions for scoring
def normalize(series: pd.Series) -> pd.Series:
    values = pd.to_numeric(series, errors="coerce").fillna(0)
    minimum = values.min()
    maximum = values.max()
    if maximum == minimum:
        return pd.Series(np.zeros(len(values)), index=values.index)
    return (values - minimum) / (maximum - minimum)

def percentile_rank(series: pd.Series) -> pd.Series:
    values = pd.to_numeric(series, errors="coerce").fillna(0)
    return values.rank(method="average", pct=True).fillna(0)

# 1. Define rule flags
stale_flag = (df["days_since_last_update"] >= 90).astype(int)
visible_flag = (df["impressions_90d"] >= 500).astype(int)
striking_flag = ((df["avg_position"] > 3) & (df["avg_position"] <= 15)).astype(int)
rule_match = stale_flag * visible_flag * striking_flag

# 2. Calculate continuous components
freshness_risk_score = percentile_rank(df["days_since_last_update"])
visibility_score = percentile_rank(np.log1p(df["impressions_90d"]))
position_opp = 1 - normalize(df["avg_position"].clip(lower=3, upper=15))

# 3. Compute continuous score for matches, 0 for non-matches
df["baseline_score"] = rule_match * (0.4 * freshness_risk_score + 0.4 * visibility_score + 0.2 * position_opp)

# 4. Attach ONE reason code and ONE action label
df["reason_code"] = np.where(rule_match == 1, "stale_striking_visible", "none")
df["suggested_action"] = np.where(rule_match == 1, "refresh", "monitor")

# 5. Rank the pages
df["baseline_rank"] = df["baseline_score"].rank(method="first", ascending=False).astype(int)

# Sort queue by rank
queue = df.sort_values("baseline_rank").copy()

# 6. Write to CSV
output_columns = [
    "content_id",
    "client_id",
    "baseline_rank",
    "baseline_score",
    "reason_code",
    "suggested_action",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "is_declining_label"
]

output_dir = "D:/Flyrank/work/outputs"
os.makedirs(output_dir, exist_ok=True)
csv_path = os.path.join(output_dir, "baseline_action_score.csv")
queue[output_columns].to_csv(csv_path, index=False)

print(f"Total matches: {rule_match.sum()}")
print(f"Wrote baseline queue to: {csv_path}")
print(f"Top-50 decline rate (precision@50): {queue.head(50)['is_declining_label'].mean():.3f}")


Total matches: 3368
Wrote baseline queue to: D:/Flyrank/work/outputs\baseline_action_score.csv
Top-50 decline rate (precision@50): 0.420


## 3. Top-10 review

Below is the manual audit of our top 10 recommendations, specifying the suggested action, why the page is prioritized, and what would make this priority wrong.

| Rank | Content ID | Action | Why It's There | What Would Make It Wrong |
|---|---|---|---|---|
| 1 | `content_69fad7e6c50c` | `refresh` | Stale (106 days), high visibility (28,000 impressions), striking position (4.7) | If the decline is seasonal rather than content decay. |
| 2 | `content_6ac3ab740bbf` | `refresh` | Stale (106 days), high visibility (22,462 impressions), striking position (4.6) | If another page on the client's site is cannibalizing this keyword. |
| 3 | `content_8b4125af6f86` | `refresh` | Stale (105 days), high visibility (10,068 impressions), striking position (3.6) | If conversion rates are already optimal and editing risks losing current rankings. |
| 4 | `content_3430a8b94511` | `refresh` | Stale (104 days), extreme visibility (152,617 impressions), striking position (3.3) | If traffic was driven by a temporary news spike that has naturally faded. |
| 5 | `content_cbd93118300b` | `refresh` | Stale (104 days), extreme visibility (145,292 impressions), striking position (3.3) | If page load speeds or UX are the root cause rather than textual content. |
| 6 | `content_09783793b38d` | `refresh` | Stale (104 days), high visibility (66,048 impressions), striking position (3.1) | If search engines introduced new AI overview panels, permanently reducing CTR. |
| 7 | `content_399f4eed93b9` | `refresh` | Stale (104 days), extreme visibility (127,658 impressions), striking position (3.4) | If the page is evergreen product documentation that does not require textual updates. |
| 8 | `content_c1143eda3230` | `refresh` | Stale (104 days), extreme visibility (112,578 impressions), striking position (3.4) | If business priorities shifted away from the topic, making refresh effort redundant. |
| 9 | `content_dcf44a9508f2` | `refresh` | Stale (104 days), high visibility (50,238 impressions), striking position (3.1) | If broken technical features/widgets are the cause, requiring an engineering fix. |
| 10 | `content_37106924f264` | `refresh` | Stale (104 days), high visibility (89,311 impressions), striking position (3.4) | If the keyword search volume collapsed globally across the web. |


In [3]:
top_10 = queue.head(10)[[
    "baseline_rank", "content_id", "baseline_score", 
    "days_since_last_update", "impressions_90d", 
    "avg_position", "is_declining_label"
]]
print("Programmatic printout of Top 10 Queue:")
print(top_10.to_string(index=False))


Programmatic printout of Top 10 Queue:
 baseline_rank           content_id  baseline_score  days_since_last_update  impressions_90d  avg_position  is_declining_label
             1 content_69fad7e6c50c        0.952187                     106            28000           4.7                   1
             2 content_6ac3ab740bbf        0.949220                     106            22462           4.6                   1
             3 content_8b4125af6f86        0.938080                     105            10068           3.6                   0
             4 content_3430a8b94511        0.931547                     104           152617           3.3                   0
             5 content_cbd93118300b        0.931373                     104           145292           3.3                   1
             6 content_09783793b38d        0.930853                     104            66048           3.1                   1
             7 content_399f4eed93b9        0.929267                     

## 4. Weak picks + leakage check

### Weak Picks & The Stability Paradox
By evaluating our top-50 matches against bottom matches, we find an interesting pattern. The top 50 matches represent traffic champions that occupy page 1 (mean position 3.86) with huge volume (~94.8k impressions). Their observed decline rate is 40.0%, which is lower than the base rate of decline (54.2%) and much lower than the bottom matches (70.0%).

This represents a **stability paradox**: the most visible, top-ranking pages are structurally the most stable, meaning a rule that prioritizes them will show lower precision@50 against decline. However, when these major pages do slip, the absolute search traffic loss is massive compared to a low-volume page, which justifies their priority for refresh review.

### Leakage Check
We confirm that no future-window or label-derived variables (such as `trend_pct`, `trend_direction`, or `is_declining_label`) were used in our feature set, rule criteria, or baseline score calculation. All features represent purely historical, decision-time measurements, preserving the integrity of our baseline.


In [4]:
top_50 = queue.head(50)
bottom_matches = queue.iloc[3300:3350] # Matches at the tail of the matching queue

print("Top 50 Matches Statistics:")
print(f"  Mean Impressions: {top_50['impressions_90d'].mean():.1f}")
print(f"  Mean Position: {top_50['avg_position'].mean():.2f}")
print(f"  Mean Days since update: {top_50['days_since_last_update'].mean():.1f}")
print(f"  Decline Rate: {top_50['is_declining_label'].mean():.3f}\n")

print("Bottom Matches (ranks 3300-3350) Statistics:")
print(f"  Mean Impressions: {bottom_matches['impressions_90d'].mean():.1f}")
print(f"  Mean Position: {bottom_matches['avg_position'].mean():.2f}")
print(f"  Mean Days since update: {bottom_matches['days_since_last_update'].mean():.1f}")
print(f"  Decline Rate: {bottom_matches['is_declining_label'].mean():.3f}")


Top 50 Matches Statistics:
  Mean Impressions: 85496.9
  Mean Position: 3.55
  Mean Days since update: 104.1
  Decline Rate: 0.420

Bottom Matches (ranks 3300-3350) Statistics:
  Mean Impressions: 686.2
  Mean Position: 13.86
  Mean Days since update: 104.0
  Decline Rate: 0.680


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.